# QuantAI specialist model — training and evaluation

This notebook fine-tunes an open-weight instruction model for QuantAI using LoRA/QLoRA. It does **not** call a paid LLM chat API. The resulting model still depends on its chosen pretrained base; this is specialised fine-tuning, not pretraining from scratch.

**Recommended:** choose a GPU runtime before starting. Training can fail or run very slowly without a GPU. Free GPU availability and session duration vary. The starter dataset is a pipeline smoke-test dataset, not a publication-quality corpus; review and expand it before treating results as reliable.

No Hugging Face access token is needed to download a public base model or train locally. Do not add secrets or private research data to this notebook.

In [ ]:
import os, subprocess, pathlib, torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'QuantAI training needs a GPU runtime. In Colab, choose Runtime > Change runtime type > T4 GPU (if available) > Save, then reconnect and run all cells again. '
        'This notebook cannot turn a GPU on programmatically. Stop here rather than spending time installing packages for a CPU-only training run.'
    )
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', torch.cuda.get_device_properties(0).total_memory // (1024**3), 'GiB')
REPO = '/content/model-analysis--hub'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','--branch','quantai-action-completeness-v2','https://github.com/kamaldeenjnr/model-analysis--hub.git',REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin','quantai-action-completeness-v2'], check=True)
    subprocess.run(['git','-C',REPO,'checkout','-B','quantai-action-completeness-v2','FETCH_HEAD'], check=True)
os.chdir(os.path.join(REPO,'quantai-model'))
print('Working directory:', os.getcwd())

In [ ]:
# Remove unrelated preinstalled packages that require Hugging Face Hub 1.x.
# QuantAI's Transformers 4.x training stack uses Hub 0.x; these packages are not used here.
%pip uninstall -y gradio gradio-client diffusers torchao


In [ ]:
# Install only QuantAI's declared training/inference dependencies.
%pip install -q -r requirements.txt


## 1. Validate the data

This validates file structure, supported actions, minimum counts and exact prompt separation. It does not establish that the examples are scientifically correct. Review training and evaluation examples before a serious run.

In [ ]:
!python validate_data.py --train data/train.jsonl --eval data/eval.jsonl
!pytest -q test_local_service.py test_frontend_contract.py

## 2. Train the QuantAI adapter

This uses the Qwen3 4B Instruct base and LoRA/4-bit loading when the environment supports it. This is a practical starting point for limited compute, not a claim that a small model has expert-level performance. If the GPU runs out of memory, reduce `--max-length` and batch settings or use a smaller base. Keep the held-out evaluation file unchanged during training.

In [ ]:
assert torch.cuda.is_available(), 'A GPU runtime is strongly recommended; stop here rather than unknowingly running a very slow CPU training job.'
!python train_lora.py --train data/train.jsonl --eval data/eval.jsonl --output outputs/quantai-qwen3-4b-lora-v3 --epochs 3 --max-length 1024

## 3. Evaluate against the untouched base model

Review the JSON reports and individual failures. Aggregate scores on a small synthetic set are only a smoke test. Do not publish performance claims from these fixtures.

In [ ]:
!python evaluate.py --model outputs/quantai-qwen3-4b-lora-v3 --cases data/eval.jsonl --report outputs/fine-tuned-evaluation-v3.json
!python evaluate.py --model outputs/quantai-qwen3-4b-lora-v3 --baseline-only --cases data/eval.jsonl --report outputs/base-evaluation-v3.json
import json
from pathlib import Path
base = json.loads(Path('outputs/base-evaluation-v3.json').read_text())
tuned = json.loads(Path('outputs/fine-tuned-evaluation-v3.json').read_text())
metrics = ['valid_json_rate', 'valid_schema_rate', 'supported_action_rate', 'action_contract_valid_rate', 'action_type_accuracy', 'exact_action_match_rate']
print(f"{'Metric':32} {'Base':>10} {'QuantAI v3':>12} {'Change':>11}")
for metric in metrics:
    b, t = base[metric], tuned[metric]
    print(f'{metric:32} {b:9.1%} {t:11.1%} {(t-b)*100:+9.2f} pp')
print('\nNon-exact action cases (for targeted debugging):')
for row in tuned['results']:
    if row['exact_action_match']:
        continue
    expected = row['expected'].get('action')
    actual = (row.get('parsed') or {}).get('action')
    print('\nPrompt:', row['prompt'])
    print('Expected action:', json.dumps(expected, ensure_ascii=False))
    print('Actual action:  ', json.dumps(actual, ensure_ascii=False))


## 4. Export a standalone model

The merged model includes the base weights and trained adapter. It can run locally without a per-request inference API, but it requires enough RAM/VRAM and storage. Check base-model licensing before redistributing weights.

In [ ]:
!python export_model.py --adapter outputs/quantai-qwen3-4b-lora-v3 --output outputs/quantai-merged-model-v3
print('Export exists:', pathlib.Path('outputs/quantai-merged-model-v3').exists())

## 5. Optional: persist output to Google Drive

Mount Drive only if you want to keep the model after the notebook session. The merged model may take several gigabytes. Do not share a Drive folder containing private data or credentials.

In [ ]:
# Uncomment and run when you want to save the v3 model to your own Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# target='/content/drive/MyDrive/QuantAI/quantai-merged-model-v3'
# os.makedirs(os.path.dirname(target), exist_ok=True)
# shutil.copytree('outputs/quantai-merged-model-v3', target, dirs_exist_ok=True)
# print('Saved to', target)


## Release checklist

- [ ] Review all training targets and the full held-out evaluation set.
- [ ] Compare base and fine-tuned failures by category.
- [ ] Expand the dataset with licensed, reviewed material and expert-authored examples.
- [ ] Test hallucination, prompt injection, missing variables, complex survey design and unsupported requests.
- [ ] Verify the local server on the target hardware and benchmark latency/memory.
- [ ] Keep statistical calculations in the deterministic engine and validate every requested action.
- [ ] Do not claim production readiness until independent review and acceptance thresholds pass.